# Point Cloud Segmentation — Colab GPU Runner

This notebook clones the repo, installs deps, and launches `train.py` on Colab's GPU.
All training logic lives in the repo — this file is just the launch wrapper.

**Before running:** set your W&B API key in Colab Secrets (🔑 icon in the left sidebar) with the name `WANDB_API_KEY`.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/amrits-creations/pcseg.git
%cd pcseg

## 2. Install dependencies

Colab already has a CUDA-enabled PyTorch, but we reinstall the pinned version
to match `requirements.txt` exactly.

In [ ]:
# Install the pinned CUDA build of torch first (same version as local, different wheel)
!pip install torch==2.12.1 --quiet
# Install everything else from requirements.txt
!pip install -r requirements.txt --quiet

## 3. Authenticate W&B via Colab Secrets

The key is read from Colab's secure secret store — it never appears in the notebook.
If the secret can't be read (it flakes occasionally), the cell falls back to a **hidden prompt** so you can paste the key without re-running the whole notebook.

In [ ]:
import os
from getpass import getpass

# Primary path: read the key from Colab Secrets (🔑 sidebar).
key = None
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
except Exception as e:
    print(f"⚠️  Couldn't read the key from Colab Secrets ({type(e).__name__}). Falling back to manual entry.")

# Fallback: prompt for the key in the cell output. getpass() hides the
# input so the key never gets echoed into the saved notebook.
if not key:
    key = getpass("Paste your W&B API key (hidden input): ").strip()

os.environ["WANDB_API_KEY"] = key
print("✅ WANDB_API_KEY is set." if key else "❌ No key set — training will fail to log.")

## 4. Download the dataset

See the README "Dataset" section for the exact wget command.
If you've already downloaded and unzipped to Google Drive, mount it instead.

In [ ]:
# Option A — download fresh (slow, ~5 GB)
# !wget -q https://cvg-data.inf.ethz.ch/s3dis/Stanford3dDataset_v1.2_Aligned_Version.zip -O s3dis.zip
# !unzip -q s3dis.zip -d data/
# !python -m pcseg.preprocessing

# Option B — mount from Google Drive (fast if already downloaded)
# from google.colab import drive
# drive.mount('/content/drive')
# !ln -s /content/drive/MyDrive/pcseg_data data

## 5. Smoke test (optional — confirms the pipeline runs before the full GPU run)

In [ ]:
!python scripts/train.py --smoke

## 6. Train

In [ ]:
!python scripts/train.py --epochs 32 --batch-size 16 --device auto